# Self-Driving Lab Inference on H100 With Unsloth

This notebook loads a quantized Unsloth model, builds the same self-driving lab observation prompt used during training, generates the next structured lab action, and steps the simulator in a short closed-loop rollout similar to `run_agent.py`, but with faster 4-bit inference on H100.

In [ ]:
%pip install -q -U torch transformers unsloth

In [ ]:
import json

import torch

from training_script import format_observation
from training_unsloth import generate_action_with_model, load_model_artifacts
from server.hackathon_environment import BioExperimentEnvironment

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
MODEL_PATH = "artifacts/grpo-unsloth-output"  # or a Hugging Face repo / base model id
SCENARIO_NAME = "cardiac_disease_de"
SEED = 42

tokenizer, model = load_model_artifacts(
    MODEL_PATH,
    trust_remote_code=True,
    max_seq_length=2048,
    load_in_4bit=True,
    fast_inference=True,
    prepare_for_inference=True,
)

env = BioExperimentEnvironment(scenario_name=SCENARIO_NAME, domain_randomise=False)
obs = env.reset(seed=SEED)
print(format_observation(obs)[:3000])

In [ ]:
result = generate_action_with_model(
    model,
    tokenizer,
    obs,
    max_new_tokens=160,
    temperature=0.2,
    top_p=0.9,
    do_sample=True,
)

print("Model response:\n")
print(result["response_text"])
print("\nParsed action:\n")
result["action"].model_dump() if result["action"] is not None else None

In [ ]:
if result["action"] is not None:
    next_obs = env.step(result["action"])
    print("Reward:", next_obs.reward)
    print("Done:", next_obs.done)
    print("Violations:", next_obs.rule_violations)
    print("Markers:", next_obs.discovered_markers[:5])
    print("Mechanisms:", next_obs.candidate_mechanisms[:5])
    if next_obs.latest_output is not None:
        print("Summary:", next_obs.latest_output.summary)
        print("Latest data preview:")
        print(json.dumps(next_obs.latest_output.data, indent=2)[:1200])
else:
    print("Model output did not parse into an ExperimentAction.")

In [ ]:
# Optional short closed-loop rollout.
obs = env.reset(seed=7)
trajectory = []

for step_idx in range(8):
    result = generate_action_with_model(model, tokenizer, obs, max_new_tokens=160)
    action = result["action"]
    record = {
        "step": step_idx + 1,
        "response_text": result["response_text"],
        "action": action.model_dump() if action is not None else None,
    }
    trajectory.append(record)
    if action is None:
        break

    next_obs = env.step(action)
    record.update({
        "reward": next_obs.reward,
        "done": next_obs.done,
        "violations": list(next_obs.rule_violations),
        "latest_summary": next_obs.latest_output.summary if next_obs.latest_output is not None else None,
        "discovered_markers": list(next_obs.discovered_markers[:5]),
        "candidate_mechanisms": list(next_obs.candidate_mechanisms[:5]),
    })
    obs = next_obs
    if obs.done:
        break

trajectory